In [211]:
import pandas as pd
from pathlib import Path
import re

pd.set_option('display.max_colwidth', None)
def load_data(file_path):
    """
    Load data from a CSV file.

    Parameters:
    file_path (str): The path to the CSV file.

    Returns:
    pd.DataFrame: A DataFrame containing the loaded data.
    """
    try:
        # data = pd.read_csv(file_path, sep="|", keep_default_na=False)
        data = pd.read_csv(file_path, sep="|")
        # data = pd.read_json(file_path, lines=True)
        # data = pd.read_json(file_path)
        # data = pd.read_excel(file_path)
        print(f"Data loaded successfully from {file_path}")
        return data
    except Exception as e:
        print(f"An error occurred while loading the data: {e}")
        return None

filepath = Path("/home/user/Downloads/2026-08-19/DataHut_AU_Coles_FullDump_20260819.CSV")   
loaded_data = load_data(filepath)

print("\nDATASET SHAPE")
print("--------------------------------")
print("The shape =", loaded_data.shape)

/tmp/ipykernel_6025/943082552.py:18: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path, sep="|")


Data loaded successfully from /home/user/Downloads/2026-08-19/DataHut_AU_Coles_FullDump_20260819.CSV

DATASET SHAPE
--------------------------------
The shape = (761163, 126)


In [185]:
loaded_data.columns.tolist()

['unique_id',
 'competitor_name',
 'store_name',
 'store_addressline1',
 'store_addressline2',
 'store_suburb',
 'store_state',
 'store_postcode',
 'store_addressid',
 'extraction_date',
 'product_name',
 'brand',
 'brand_type',
 'grammage_quantity',
 'grammage_unit',
 'drained_weight',
 'producthierarchy_level1',
 'producthierarchy_level2',
 'producthierarchy_level3',
 'producthierarchy_level4',
 'producthierarchy_level5',
 'producthierarchy_level6',
 'producthierarchy_level7',
 'regular_price',
 'selling_price',
 'price_was',
 'promotion_price',
 'promotion_valid_from',
 'promotion_valid_upto',
 'promotion_type',
 'percentage_discount',
 'promotion_description',
 'package_sizeof_sellingprice',
 'per_unit_sizedescription',
 'price_valid_from',
 'price_per_unit',
 'multi_buy_item_count',
 'multi_buy_items_price_total',
 'currency',
 'breadcrumb',
 'pdp_url',
 'variants',
 'product_description',
 'instructions',
 'storage_instructions',
 'preparationinstructions',
 'instructionforuse',


In [305]:
loaded_data['unique_id'].duplicated().sum()
# print(loaded_data.loc[
#     loaded_data['unique_id'].duplicated(keep=False),
#     'unique_id'
# ].unique()) 

733520

In [347]:
loaded_data['store_name'].unique().tolist()

['GLADSTONE',
 'CANNONVALE',
 'DEE WHY',
 'RUNAWAY BAY',
 'WILSONTON',
 'ALEXANDER HEIGHTS',
 'CHRISTIES BEACH',
 'HOPPERS CROSSING',
 'SUNNYBANK HILLS',
 'BATEAU BAY',
 'HASTINGS',
 'GREENACRE',
 'WEST GOSFORD',
 'WAGGA WAGGA',
 'SPRINGFIELD',
 'ROUSE HILL',
 'CANBERRA',
 'LAKELANDS',
 'WODONGA',
 'RIVERTON',
 'CHURCHILL',
 'Coles Local Manly Corso',
 'Coles Local Surrey Hills',
 'Coles Local Woolloongabba']

In [342]:
loaded_data['file_name_1'].isnull().sum()

761163

In [361]:
# General check
# 'energijska vrednost', 'maščobe', '— nasičene maščobe', 'ogljikovi hidrati', '— sladkorji', 'beljakovine'
field = 'storage_instructions'
empty_field_records = loaded_data[
    (loaded_data[field].notna())
    & 
    # loaded_data['percentage_discount'].isna() 
    # loaded_data['promotion_price'].notna()
    # &
    # (loaded_data['promotion_valid_from'].isna() | loaded_data['promotion_valid_from'].isna())
    # (loaded_data[field] != loaded_data['selling_price'])
    # (pd.to_numeric(loaded_data[field], errors="coerce").isna())
    # &
    # (loaded_data[field].astype(str).str.contains('...', na=False))
    # (loaded_data[field].astype(str).str.strip() != 'MultiBuy') 
    # & 
    # (loaded_data['promotion_price'] != loaded_data['selling_price'])
    (loaded_data[field] == '.')
    #  loaded_data[field].astype(str).str.strip().str.lower().eq('(null)')
    # &
    # (pd.to_numeric(loaded_data[field], errors='coerce').isna() <= 0) 
  
]

# Print unique_id and pdp_url
# print(empty_field_records)
print(empty_field_records[[field, 'pdp_url']])

       storage_instructions  \
376248                    .   
376249                    .   
376250                    .   
376251                    .   
376252                    .   
376253                    .   
376254                    .   
376255                    .   
376256                    .   
376257                    .   
376258                    .   
376259                    .   
376260                    .   
376261                    .   
376262                    .   
376263                    .   
376264                    .   
376265                    .   
376266                    .   
376267                    .   
376268                    .   
491922                    .   
491923                    .   
491924                    .   
491925                    .   
491926                    .   
491927                    .   
491928                    .   
491929                    .   
491930                    .   
491931                    .   
491932  

In [213]:
# CSV null summary
# Check whether each column is unique (ignoring NaN and empty strings)


null_summary = pd.DataFrame({
    'Null_Count': loaded_data.isnull().sum(),
    'Null_Percentage': (loaded_data.isnull().sum() / len(loaded_data)) * 100,
    'Unique': loaded_data.apply(lambda col: "Yes" if col.is_unique else "No"),
    'Value': loaded_data.apply(lambda col: col.unique() if col.nunique() <= 2 else "More values")
})

# Round percentage to 2 decimal places
null_summary['Null_Percentage'] = null_summary['Null_Percentage'].round(2)

print(null_summary)

# Print column names with 100% null values
completely_null_columns = null_summary.index[null_summary['Null_Percentage'] == 100].tolist()
print("\nColumns with 100% null values:")
print(completely_null_columns)


# mandatory_columns = null_summary.index[null_summary['Null_Percentage'] == 0].tolist()
# print("\nColumns with 0 null values:")
# print(mandatory_columns)


                    Null_Count  Null_Percentage Unique        Value
unique_id                    0             0.00     No  More values
competitor_name              0             0.00     No      [coles]
store_name                   0             0.00     No  More values
store_addressline1      761163           100.00     No        [nan]
store_addressline2      761163           100.00     No        [nan]
...                        ...              ...    ...          ...
suitable_for            761163           100.00     No        [nan]
standard_drinks         702123            92.24     No  More values
environmental           761163           100.00     No        [nan]
grape_variety           761163           100.00     No        [nan]
retail_limit              7286             0.96     No  More values

[126 rows x 4 columns]

Columns with 100% null values:
['store_addressline1', 'store_addressline2', 'store_suburb', 'drained_weight', 'producthierarchy_level6', 'producthierarchy_leve

In [250]:
list1 = ['store_name', 'store_addressline1', 'store_addressline2', 'store_suburb', 'store_state', 'store_postcode', 'store_addressid', 'brand_type', 'drained_weight', 'producthierarchy_level5', 'producthierarchy_level6', 'producthierarchy_level7', 'promotion_type', 'package_sizeof_sellingprice', 'per_unit_sizedescription', 'multi_buy_item_count', 'multi_buy_items_price_total', 'variants', 'instructions', 'storage_instructions', 'preparationinstructions', 'instructionforuse', 'country_of_origin', 'allergens', 'age_of_the_product', 'age_recommendations', 'flavour', 'nutritions', 'nutritional_information', 'vitamins', 'labelling', 'grade', 'region', 'packaging', 'receipies', 'processed_food', 'barcode', 'frozen', 'chilled', 'organictype', 'cooking_part', 'handmade', 'max_heating_temperature', 'special_information', 'label_information', 'dimensions', 'special_nutrition_purpose', 'feeding_recommendation', 'warranty', 'color', 'model_number', 'material', 'usp', 'dosage_recommendation', 'tasting_note', 'food_preservation', 'size', 'file_name_1', 'file_name_2', 'file_name_3', 'file_name_4', 'file_name_5', 'file_name_6', 'competitor_product_key', 'fit_guide', 'occasion', 'material_composition', 'style', 'care_instructions', 'heel_type', 'heel_height', 'upc', 'features', 'dietary_lifestyle', 'manufacturer_address', 'importer_address', 'distributor_address', 'vinification_details', 'recycling_information', 'return_address', 'alchol_by_volume', 'beer_deg', 'netcontent', 'netweight', 'ingredients', 'random_weight_flag', 'promo_limit', 'multibuy_items_pricesingle', 'perfect_match', 'servings_per_pack', 'warning', 'suitable_for', 'standard_drinks', 'environmental', 'grape_variety', 'retail_limit']
list2 = ['store_name','store_addressline1','store_addressline2','store_suburb','store_state','store_postcode','store_addressid','brand_type','drained_weight','producthierarchy_level5','producthierarchy_level6','producthierarchy_level7','promotion_type','package_sizeof_sellingprice','per_unit_sizedescription','multi_buy_item_count','multi_buy_items_price_total','variants','instructions','storage_instructions','preparationinstructions','instructionforuse','allergens','age_of_the_product','age_recommendations','flavour','nutritions','nutritional_information','vitamins','labelling','grade','region','packaging','receipies','processed_food','barcode','frozen','chilled','organictype','cooking_part','handmade','max_heating_temperature','special_information','label_information','dimensions','special_nutrition_purpose','feeding_recommendation','warranty','color','model_number','material','usp','dosage_recommendation','tasting_note','food_preservation','size','competitor_product_key','fit_guide','occasion','material_composition','style','care_instructions','heel_type','heel_height','upc','features','dietary_lifestyle','manufacturer_address','importer_address','distributor_address','vinification_details','recycling_information','return_address','alchol_by_volume','beer_deg','netcontent','netweight','ingredients','random_weight_flag','promo_limit','multibuy_items_pricesingle','perfect_match','servings_per_pack','warning','suitable_for','standard_drinks','environmental','grape_variety','retail_limit','file_name_1','file_name_2','file_name_3','file_name_4','file_name_5','file_name_6']
print(set(list1)-set(list2))

{'country_of_origin'}


In [379]:
# JSON null summary


# Count NaN or empty string as null
null_count = loaded_data.apply(
    lambda col: col.isna().sum() + (col.astype(str).str.strip() == "").sum()
)

# Check whether each column is unique (ignoring NaN and empty strings)
# unique_check = loaded_data.apply(
#     lambda col: "Yes" if col.replace("", pd.NA).dropna().is_unique else "No"
# )

# Create summary
null_summary = pd.DataFrame({
    "Null_Count": null_count,
    "Null_Percentage": (null_count / len(loaded_data) * 100).round(2),
    # "Unique": unique_check
})

print(null_summary)

# Columns with 100% null/empty values
completely_null_columns = null_summary.index[
    null_summary["Null_Percentage"] == 100
].tolist()

print("\nColumns with 100% null/empty values:")
print(completely_null_columns)

                      Null_Count  Null_Percentage
reference_number          235813           100.00
id                             0             0.00
url                            0             0.00
broker_display_name         9233             3.92
broker                      9233             3.92
category                       0             0.00
category_url                   0             0.00
title                          0             0.00
description                    1             0.00
location                       0             0.00
price                          0             0.00
currency                       0             0.00
price_per                 235813           100.00
bedrooms                   27891            11.83
bathrooms                  27895            11.83
furnished                      0             0.00
rera_permit_number        235813           100.00
dtcm_licence              235813           100.00
scraped_ts                     0             0.00


In [282]:
violations = []

for col in loaded_data.columns:
    for idx, value in loaded_data[col].items():
        # if value == 'Null' :
        if value == ' ' :
        # if value == {} or value == '[]' or value == 'Nan'or value == 'NaN'or value == 'Null'or value == 'null' or value == ' ' or '\n' in str(value) or r'\n' in str(value) or '\t' in str(value) or r'\t' in str(value) or '\r' in str(value) or r'\r' in str(value) :
            violations.append({
                "row_index": idx,
                "column": col,
                "value": value
            })
violations_df = pd.DataFrame(violations)

print(f"Number of cells containing {{}} or []: {len(violations_df)}")
print(violations_df)

Number of cells containing {} or []: 0
Empty DataFrame
Columns: []
Index: []


In [362]:
# import pandas as pd
# import re

# # Columns to validate
columns_to_check = [
# 'product_name',
# 'promotion_description'
'product_description',
#  'instructions',
#  'storage_instructions',
#  'preparationinstructions',
#  'instructionforuse'
# 'description'
# 'breadcrumb'
]

# # Check whether requested columns exist
missing_columns = [
    col for col in columns_to_check
    if col not in loaded_data.columns
]

if missing_columns:
    print(
        f"ERROR: The following columns do not exist in the dataframe: "
        f"{missing_columns}"
    )

# Process only columns that exist
existing_columns = [
    col for col in columns_to_check
    if col in loaded_data.columns
]

invalid_records = []

for col in existing_columns:

    # Convert values to string
    values = loaded_data[col].astype('string')

    # Ignore null and empty/whitespace-only values
    non_empty = (
        values.notna() &
        values.str.strip().ne("")
    )

    # Check invalid patterns only for non-empty values
    invalid_mask = non_empty & (
        # Leading or trailing whitespace
        values.str.match(r'^\s+|\s+$', na=False)

        # Repeated characters 4 or more times
        | values.str.contains(
            r'(.)\1{3,}',
            regex=True,
            na=False
        )

        # Repeated X characters
        | values.str.contains(
            r'X{3,}',
            case=False,
            regex=True,
            na=False
        )

        # Contains only special characters
        | values.str.match(
            r'^[^A-Za-z0-9]+$',
            na=False
        )
    )

    # Store invalid records
    invalid_data = loaded_data.loc[
        invalid_mask,
        ['pdp_url', col]
    ].copy()

    invalid_data['invalid_column'] = col

    invalid_records.append(invalid_data)


# Combine results
if invalid_records:

    invalid_summary = pd.concat(
        invalid_records,
        ignore_index=True
    )

    print(
        f"\nNumber of invalid records: "
        f"{len(invalid_summary)}"
    )

    print(
        invalid_summary[
            ['pdp_url', 'invalid_column']
            + existing_columns
        ]
    )

else:
    print("\nNo invalid values found.")

/tmp/ipykernel_6025/4286128344.py:54: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | values.str.contains(



Number of invalid records: 2946
                                                                           pdp_url  \
0       https://www.coles.com.au/product/xxxx-ginger-beer-can-330ml-6-pack-1282826   
1       https://www.coles.com.au/product/xxxx-ginger-beer-can-330ml-6-pack-1282826   
2       https://www.coles.com.au/product/xxxx-ginger-beer-can-330ml-6-pack-1282826   
3       https://www.coles.com.au/product/xxxx-ginger-beer-can-330ml-6-pack-1282826   
4       https://www.coles.com.au/product/xxxx-ginger-beer-can-330ml-6-pack-1282826   
...                                                                            ...   
2941  https://www.coles.com.au/product/coles-kitchen-chicken-leek-pie-350g-5587111   
2942  https://www.coles.com.au/product/coles-kitchen-chicken-leek-pie-350g-5587111   
2943  https://www.coles.com.au/product/coles-kitchen-chicken-leek-pie-350g-5587111   
2944  https://www.coles.com.au/product/coles-kitchen-chicken-leek-pie-350g-5587111   
2945  https://www.col

In [343]:
# white space checking
for col in loaded_data.columns.tolist():
    if loaded_data[col].nunique() > 1 :

        column = col   # Replace with your column name

        # Find rows with leading/trailing spaces or multiple spaces between words
        invalid_rows = loaded_data[
            loaded_data[column].fillna('').astype(str).str.contains(
                r'(^\s)|(\s$)|(\s{2,})',
                regex=True
            )
        ]
        print(col)
        print("Number of invalid rows:", len(invalid_rows))
        print(invalid_rows[['pdp_url', column]].head(10))
        print(len(invalid_rows))

/tmp/ipykernel_6025/350724292.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  loaded_data[column].fillna('').astype(str).str.contains(


unique_id
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, unique_id]
Index: []
0


/tmp/ipykernel_6025/350724292.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  loaded_data[column].fillna('').astype(str).str.contains(


store_name
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, store_name]
Index: []
0


/tmp/ipykernel_6025/350724292.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  loaded_data[column].fillna('').astype(str).str.contains(


...

perfect_match
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, perfect_match]
Index: []
0


/tmp/ipykernel_6025/350724292.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  loaded_data[column].fillna('').astype(str).str.contains(


warning
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, warning]
Index: []
0
standard_drinks
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, standard_drinks]
Index: []
0


/tmp/ipykernel_6025/350724292.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  loaded_data[column].fillna('').astype(str).str.contains(
/tmp/ipykernel_6025/350724292.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  loaded_data[column].fillna('').astype(str).str.contains(


retail_limit
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, retail_limit]
Index: []
0


In [197]:
# checking value ends with ','
for col in loaded_data.columns.tolist():
    if loaded_data[col].nunique() > 1 :

        column = col   # Replace with your column name

        # Find rows with leading/trailing spaces or multiple spaces between words
        invalid_rows = loaded_data[
            loaded_data[column].fillna('').astype(str).str.endswith(',') |
            loaded_data[column].fillna('').astype(str).str.endswith('|') |
            loaded_data[column].fillna('').astype(str).str.endswith(':') |
            loaded_data[column].fillna('').astype(str).str.endswith(';') |
            loaded_data[column].fillna('').astype(str).str.endswith('>') |
            loaded_data[column].fillna('').astype(str).str.endswith(' ') |
            (loaded_data[column].astype(str).str.strip().str.lower().eq('null')) |
            (loaded_data[column].astype(str).str.strip().str.lower().eq('nan'))
            # loaded_data[column].astype(str).str.contains(r'\n', regex=False, na=False)
            # loaded_data[column].fillna('').astype(str).str.contains("'\n'", regex=False, na=False) |
            # loaded_data[column].fillna('').astype(str).str.contains("'\t'", regex=False, na=False) |
            # loaded_data[column].fillna('').astype(str).str.contains("'\r'", regex=False, na=False)
            # (loaded_data[column].astype(str).str.strip()  == 'nan')
        ]
        print(col)
        print("Number of invalid rows:", len(invalid_rows))
        print(invalid_rows[['pdp_url', column]].head(10))
        print(len(invalid_rows))

unique_id
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, unique_id]
Index: []
0
store_name
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, store_name]
Index: []
0
store_state
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, store_state]
Index: []
0
store_postcode
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, store_postcode]
Index: []
0
store_addressid
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, store_addressid]
Index: []
0
product_name
Number of invalid rows: 0
Empty DataFrame
Columns: [pdp_url, product_name]
Index: []
0
brand
Number of invalid rows: 48
                                                                                        pdp_url  \
450036  https://www.coles.com.au/product/nan-expertpro-formula-lactose-intolerence-400g-6179551   
450037  https://www.coles.com.au/product/nan-expertpro-formula-lactose-intolerence-400g-6179551   
450038  https://www.coles.com.au/product/nan-expertpro-formula-lactose-i

In [127]:
# Numeric value check for specific columns
numneric_columns = [
    "unique_id",
    "selling_price",
    "regular_price",
    "promotion_price",
    "grammage_quantity",
    "percentage_discount"
    # "object_id",
    # 'Part number', 'Net price', 'Catalogue price'
    # "barcode",
    # "rating",
    # "review",
    # "user_id",
    # "bedrooms",
    # "bathrooms",
    # "number_of_photos",
    # "latitude",
    # "longitude",
    # "listing_id",
    # "rating"
    # "min_area_in_sqft", "max_area_in_sqft"
    # "depth"
    # 'count_of_unit_types', 'min_area_in_sqft', 'max_area_in_sqft'
    # 'builtup_area_sqm', 'plot_area_sqm', 'transaction_amount', 'latitude', 'longitude', 'bayut_leaf_location_id', 'transaction_per_sqm_amount', 'balcony_area_sqm'
]
for col in numneric_columns:
    # Convert to string and remove leading/trailing spaces
    values = loaded_data[col].fillna("").astype(str).str.strip()

    # Find non-empty values that cannot be converted to numeric
    invalid_rows = loaded_data[
        (values != "") &
        (pd.to_numeric(values, errors="coerce").isna())
    ]

    print(f"\nColumn: {col}")
    print(f"Number of non-numeric values: {len(invalid_rows)}")

    if not invalid_rows.empty:
        print(invalid_rows[[ "pdp_url", col]])


Column: unique_id
Number of non-numeric values: 0

Column: selling_price
Number of non-numeric values: 0

Column: regular_price
Number of non-numeric values: 0

Column: promotion_price
Number of non-numeric values: 0

Column: grammage_quantity
Number of non-numeric values: 0

Column: percentage_discount
Number of non-numeric values: 0


In [107]:

# Invalid phone numbers:
# - Ignore empty/null values
# - Invalid if non-empty and contains characters other than '+' and digits

invalid_phone_numbers = loaded_data[
    loaded_data['phone_number'].notna() &
    (loaded_data['phone_number'].astype(str).str.strip() != '') &
    ~loaded_data['phone_number'].astype(str).str.fullmatch(r'\+?\d+', na=False)
]

print(invalid_phone_numbers[['phone_number', 'url']])

Empty DataFrame
Columns: [phone_number, url]
Index: []


In [137]:
# Date validation for specific columns
import pandas as pd

# Date columns to validate
date_columns = [
    # "scraped_ts", "published_at", "date"
    # 'extraction_date',
    'promotion_valid_from',
    'promotion_valid_upto',
    'price_valid_from'
    # 'published_at'
]

# Check whether columns exist
missing_columns = [
    col for col in date_columns
    if col not in loaded_data.columns
]

if missing_columns:
    print(
        f"ERROR: The following date columns do not exist in the dataframe: "
        f"{missing_columns}"
    )

# Process only existing columns
existing_date_columns = [
    col for col in date_columns
    if col in loaded_data.columns
]

invalid_date_records = []

for col in existing_date_columns:

    # Convert values to string
    values = loaded_data[col].astype('string')

    # Ignore empty/null values
    non_empty = (
        values.notna() &
        values.str.strip().ne("")
    )

    # Validate exact YYYY-MM-DD format
    invalid_mask = non_empty & ~values.str.match(
        r'^\d{2}.\d{2}.\d{4}$',
        # r'^\d{2}.\d{2}.\d{4}$',
        na=False
    )

    invalid_data = loaded_data.loc[
        invalid_mask,
        ['unique_id', col] if 'unique_id' in loaded_data.columns else [col]
    ].copy()

    invalid_data['invalid_column'] = col

    invalid_date_records.append(invalid_data)


# Combine all invalid records
if invalid_date_records:

    invalid_date_summary = pd.concat(
        invalid_date_records,
        ignore_index=True
    )

    print(
        f"\nNumber of invalid date values: "
        f"{len(invalid_date_summary)}"
    )

    print(invalid_date_summary)

else:
    print("\nNo invalid date values found.")


Number of invalid date values: 0
Empty DataFrame
Columns: [unique_id, promotion_valid_from, invalid_column, promotion_valid_upto, price_valid_from]
Index: []


In [341]:
# URL validation
from urllib.parse import urlparse
url = 'image_url_1'
id_val = 'unique_id'
# Check whether URL column exists
if url not in loaded_data.columns:
    print(f"ERROR: {url} column does not exist in the dataframe.")

else:

    # Convert URL values to string
    urls = loaded_data[url].astype('string')

    # Empty values are ignored
    non_empty = (
        urls.notna() &
        urls.str.strip().ne("")
    )

    # URL validation function
    def is_valid_url(url):
        try:
            parsed = urlparse(url)
            return (
                parsed.scheme in ['http', 'https'] and
                bool(parsed.netloc)
            )
        except Exception:
            return False

    # Validate URLs
    invalid_mask = non_empty & ~urls.apply(is_valid_url)

    # Get invalid rows
    invalid_urls = loaded_data.loc[
        invalid_mask,
        [id_val, url]
        if id_val in loaded_data.columns
        else [url]
    ]

    print(f"Number of invalid URLs: {len(invalid_urls)}")

    print(invalid_urls)
    # print(f"Number of invalid URLs: {invalid_mask.sum()}")

Number of invalid URLs: 0
Empty DataFrame
Columns: [unique_id, image_url_1]
Index: []


In [278]:
# description column validation

import re

def is_clean_description(text):
    if pd.isna(text):
        return True  # Ignore nulls (change if nulls are invalid)

    text = str(text)

    # # 1. Leading/trailing spaces
    # if text != text.strip():
    #     return False

    # 2. Multiple consecutive spaces
    if re.search(r'\s{2,}', text):
        return False

    # 3. HTML tags
    if re.search(r'<[^>]+>', text):
        return False

    # 4. HTML entities
    if re.search(r'&[a-zA-Z]+;', text):
        return False

    # # 5. Unwanted special characters
    # # Allows letters, numbers, whitespace, and common punctuation
    # if re.search(r'[^A-Za-z0-9\s.,!?()\-\'"/:&%]', text):
    #     return False

    return True

# Find rows with invalid descriptions
invalid_descriptions = loaded_data[
    ~loaded_data['description'].apply(is_clean_description)
]

print(f"Number of invalid descriptions: {len(invalid_descriptions)}")

print(invalid_descriptions[['profile_url', 'description']])

Number of invalid descriptions: 0
Empty DataFrame
Columns: [profile_url, description]
Index: []


In [279]:

def is_valid_address(address):
    if pd.isna(address):
        return False

    address = str(address)

    # Empty or whitespace-only
    if address.strip() == "":
        return False

    # Leading/trailing spaces
    if address != address.strip():
        return False

    # Multiple consecutive spaces
    if re.search(r"\s{2,}", address):
        return False

    # # Allow only letters (Unicode), spaces, apostrophes, and hyphens
    # if not re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿĀ-ž' -]+", address):
    #     return False

    return True

# Find invalid addresses
invalid_addresses = loaded_data[
    ~loaded_data["address"].apply(is_valid_address)
]

print(f"Number of invalid address values: {len(invalid_addresses)}")
print(invalid_addresses[["address"]])

Number of invalid address values: 2
   address
21        
74        


In [324]:
# Email validation

def is_valid_email(email):
    if pd.isna(email):
        return False

    email = str(email)

    # Empty or whitespace-only
    if email.strip() == "":
        return False

    # Leading/trailing spaces
    if email != email.strip():
        return False

    # No spaces inside the email
    if " " in email:
        return False

    # No consecutive dots
    if ".." in email:
        return False

    # Email format
    pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    return bool(re.fullmatch(pattern, email))

# Find invalid emails
invalid_emails = loaded_data[
    ~loaded_data["email"].apply(is_valid_email)
]

print(f"Number of invalid emails: {len(invalid_emails)}")
print(invalid_emails[["email", 'profile_url']])

Number of invalid emails: 0
Empty DataFrame
Columns: [email, profile_url]
Index: []


In [325]:


def is_valid_country(value):
    if pd.isna(value):
        return False

    value = str(value).strip()

    # Empty check
    if value == "":
        return True

    # Must NOT contain any digits (zipcode/number rule)
    if re.search(r"\d", value):
        return False

    # Allow only letters, spaces, hyphen, apostrophe
    if not re.fullmatch(r"[A-Za-zÀ-ÖØ-öø-ÿ\s\-']+", value):
        return False

    return True

# Find invalid countries
invalid_countries = loaded_data[
    ~loaded_data["country"].apply(is_valid_country)
]

print(f"Number of invalid country values: {len(invalid_countries)}")

print(invalid_countries[["country"]].drop_duplicates())

Number of invalid country values: 0
Empty DataFrame
Columns: [country]
Index: []


In [257]:
expected_prefix = 'https://online.eurospin.com/product/'

# Find rows where url does not start with the expected prefix
invalid_urls = loaded_data[
    ~loaded_data['pdp_url']
        .fillna('')
        .astype(str)
        .str.startswith(expected_prefix)
]

print(f"Number of invalid URLs: {len(invalid_urls)}\n")

# Print the invalid URLs
print(invalid_urls['pdp_url'])

Number of invalid URLs: 0

Series([], Name: pdp_url, dtype: object)


In [370]:
# Find rows where URL does not end with the unique_id value
violations = loaded_data[
    ~loaded_data.apply(
        lambda row: (
            pd.notna(row['url']) and
            pd.notna(row['unique_id']) and
            str(row['url']).rstrip('/').endswith(str(row['unique_id']))
        ),
        axis=1
    )
]

print(f"Number of violations: {len(violations)}\n")
print(violations[['unique_id', 'url']])

Number of violations: 1

                              unique_id  \
6  078706b2-e4ea-414b-ae2f-a05fd7944efe   

                                                                                          url  
6  https://eservices.ajmanded.ae/en/TradeLicense/License/53b3f3b6-4841-4558-8ba1-dfddb08c6c78  


In [207]:
# Find rows where unique_id is not present in url
violations = loaded_data[loaded_data.apply(
        lambda row: (
            pd.notna(row['id']) and
            pd.notna(row['url']) and
            str(row['id']).strip() in str(row['url'])
        ),
        axis=1
    )
]

print(f"Number of violations: {len(violations)}\n")

# Print the ids and corresponding urls
print(violations[['id', 'url']])

Number of violations: 2676

             id                                                      url
0     500130068  https://www.bayut.bh/en/property/details-500130068.html
1     500058235  https://www.bayut.bh/en/property/details-500058235.html
2     104405805  https://www.bayut.bh/en/property/details-104405805.html
3     105698449  https://www.bayut.bh/en/property/details-105698449.html
4     105280695  https://www.bayut.bh/en/property/details-105280695.html
...         ...                                                      ...
2671  105647143  https://www.bayut.bh/en/property/details-105647143.html
2672  105694471  https://www.bayut.bh/en/property/details-105694471.html
2673  105527407  https://www.bayut.bh/en/property/details-105527407.html
2674  105527400  https://www.bayut.bh/en/property/details-105527400.html
2675  105126858  https://www.bayut.bh/en/property/details-105126858.html

[2676 rows x 2 columns]


In [200]:
# Find profile_urls that do NOT contain 'xxxx'
invalid_urls = loaded_data[
    ~loaded_data['pdp_url']
        .fillna('')
        .astype(str)
        .str.contains('/nav/drive/store/', case=False, na=False)
]

print(f"Number of pdp_urls not containing '/nav/drive/store/': {len(invalid_urls)}\n")

# Print only the pdp_url values
print(invalid_urls['pdp_url'])

Number of pdp_urls not containing '/nav/drive/store/': 0

Series([], Name: pdp_url, dtype: object)


In [383]:
# Compare broker and broker_display_name columns for mismatches

broker = loaded_data["broker"].fillna("").astype(str).str.strip().str.lower()
broker_display = loaded_data["broker_display_name"].fillna("").astype(str).str.strip().str.lower()

# Find mismatches
mismatches = loaded_data[broker != broker_display]

print(f"Number of mismatches: {len(mismatches)}")

# Display mismatching records
print(
    mismatches[
        ["broker", "broker_display_name", "id","url"]
    ]
)

print(loaded_data["broker"].nunique())
print(loaded_data["broker_display_name"].nunique())

Number of mismatches: 0
Empty DataFrame
Columns: [broker, broker_display_name, id, url]
Index: []
6516
6738


In [384]:
loaded_data[
    loaded_data["broker_display_name"].duplicated(keep=False) == False
][["broker", "broker_display_name"]]

,broker,broker_display_name
61,MOSTAFA FARAG,Mostafa farag
63,محمد عبد المنعم,محمد عبد المنعم
65,SAMA ELKINANY,Sama Elkinany
82,MAHMOUD BARAKAT,Mahmoud Barakat
83,USER 0KK2OUمحمد صلاح,User 0kk2ouمحمد صلاح
...,...,...
235663,MOHAMED WAHBY,Mohamed Wahby
235709,نشوى يحيى,نشوى يحيى
235744,AHMED AMEEN,Ahmed Ameen
235748,TONY SAEED,Tony saeed


In [ ]:
# Rule:
# owner_association_activity_status should have a value only when broker_type == "Property Manager"

violations = loaded_data[
    (
        loaded_data["owner_association_activity_status"]
        .fillna("")
        .astype(str)
        .str.strip() != ""
    )
    &
    (
        loaded_data["broker_type"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("Property Manager")
    )
]

print(f"Number of violations: {len(violations)}")

# Print relevant columns
print(
    violations[
        [
            "unique_id",
            "broker_type",
            "owner_association_activity_status"
        ]
    ]
)

In [280]:
# find if integor value
col = 'regular_price'
url = 'pdp_url'
non_float_values = loaded_data[loaded_data[col].apply(lambda x: isinstance(x, int))]

print(non_float_values[[col, url]])

Empty DataFrame
Columns: [regular_price, pdp_url]
Index: []


In [267]:
# Check if price field is 2f format
url = 'pdp_url'
price_columns = [ 
                'regular_price',
                 'selling_price', 
                 'promotion_price'
                 ]

print("Checking price field is 2f format ','):\n")

for col in price_columns:
    if col in loaded_data.columns:


        # Values that do not have exactly 2 decimal places
        decimal_invalid = loaded_data[
            loaded_data[col].notna() &
            loaded_data[col].astype(str).str.strip().ne('') &
            ~loaded_data[col].astype(str).str.strip().str.match(r'^\d+\.\d{2}$')
        ]

        if len(decimal_invalid) > 0:
            print(f"❌ {col}: Found {len(decimal_invalid)} values not having exactly 2 decimal places")
            print(decimal_invalid[[col, url]])
        else:
            print(f"✓ {col}: All values have exactly 2 decimal places")

    else:
        print(f"⚠ {col}: Column not found in data")

    print()

Checking price field is 2f format ','):

❌ regular_price: Found 695575 values not having exactly 2 decimal places
        regular_price  \
0                 4.5   
1                 4.5   
2                 4.5   
3                 4.5   
4                 4.5   
...               ...   
761158          159.0   
761159          159.0   
761160          159.0   
761161          159.0   
761162          160.0   

                                                                                          pdp_url  
0                      https://www.coles.com.au/product/coles-simply-unsalted-butter-250g-3169879  
1                      https://www.coles.com.au/product/coles-simply-unsalted-butter-250g-3169879  
2                      https://www.coles.com.au/product/coles-simply-unsalted-butter-250g-3169879  
3                      https://www.coles.com.au/product/coles-simply-unsalted-butter-250g-3169879  
4                      https://www.coles.com.au/product/coles-simply-unsalted-butter-

In [346]:
# Check price/currency consistency: if one is present, the other must be too
print("Checking price/currency consistency:\n")

if 'price' in loaded_data.columns and 'currency' in loaded_data.columns:
    # Identify rows with data in price vs currency
    has_price = loaded_data['price'].notna() & (loaded_data['price'].astype(str).str.strip() != '')
    has_currency = loaded_data['currency'].notna() & (loaded_data['currency'].astype(str).str.strip() != '')
    
    # Find violations: price XOR currency (one true, one false)
    violations = loaded_data[has_price != has_currency]
    
    print(f"Total rows: {len(loaded_data)}")
    print(f"Rows violating rule: {len(violations)}")
    
    if len(violations) > 0:
        print("\nViolating row IDs:")
        violation_ids = violations['unique_id'].tolist() if 'unique_id' in loaded_data.columns else violations.index.tolist()
        print(violation_ids)
        
        print("\nSample violations:")
        cols_show = [c for c in ['unique_id', 'price', 'currency'] if c in loaded_data.columns]
        print(violations[cols_show].head(20).to_string(index=False))
    else:
        print("✓ All rows satisfy the rule: if price exists, currency exists and vice versa")
else:
    missing = []
    if 'price' not in loaded_data.columns:
        missing.append('price')
    if 'currency' not in loaded_data.columns:
        missing.append('currency')
    print(f"Missing columns: {missing}")

Checking price/currency consistency:

Total rows: 2343
Rows violating rule: 0
✓ All rows satisfy the rule: if price exists, currency exists and vice versa


In [225]:
count = (loaded_data['selling_price'] == 0).sum()

print(f"Number of rows with selling price = 0: {count}")

Number of rows with selling price = 0: 0


In [252]:
# Check if price fields use dot (.) instead of comma (,) as decimal separator
#price_columns = ['regular_price', 'selling_price', 'promotion_price', 'price_per_unit']
price_columns = ['price', 'Net price', 'Catalogue price', 'regular_price', 'selling_price', 'price_was', 'promotion_price', 'price_per_unit', 'grammage_quantity',
                 'item_mrp', 'item_selling_price', 'product_price', 'promo_offer_price' ]

print("Checking price field separators (should use dot '.' not comma ','):\n")

for col in price_columns:
    if col in loaded_data.columns:
        # Find rows with comma in price field
        comma_values = loaded_data[loaded_data[col].astype(str).str.contains(',', na=False)]
        invalid_values = loaded_data[pd.to_numeric(loaded_data[col], errors='coerce') <= 0]
        
        if len(comma_values) > 0:
            print(f"❌ {col}: Found {len(comma_values)} values with COMMA separator")
            print(f"   Sample: {comma_values[col].head(3).tolist()}")
        else:
            print(f"✓ {col}: All values use dot (.) or are numeric")
        if len(invalid_values) > 0:
            print(f"❌ {col}: Found {len(invalid_values)} values that are <= 0")
            print(f"   Sample: {invalid_values[col].head(3).tolist()}")
        else:
            print(f"✓ {col}: All values are greater than 0")
    else:
        print(f"⚠ {col}: Column not found in data")
    print()

Checking price field separators (should use dot '.' not comma ','):

⚠ price: Column not found in data

⚠ Net price: Column not found in data

⚠ Catalogue price: Column not found in data

✓ regular_price: All values use dot (.) or are numeric
✓ regular_price: All values are greater than 0

✓ selling_price: All values use dot (.) or are numeric
✓ selling_price: All values are greater than 0

✓ price_was: All values use dot (.) or are numeric
✓ price_was: All values are greater than 0

✓ promotion_price: All values use dot (.) or are numeric
✓ promotion_price: All values are greater than 0

✓ price_per_unit: All values use dot (.) or are numeric
✓ price_per_unit: All values are greater than 0

✓ grammage_quantity: All values use dot (.) or are numeric
✓ grammage_quantity: All values are greater than 0

⚠ item_mrp: Column not found in data

⚠ item_selling_price: Column not found in data

⚠ product_price: Column not found in data

⚠ promo_offer_price: Column not found in data



In [304]:

def is_valid(row):
    grammage_unit = str(row['grammage_unit']).strip().lower()
    site_shown_uom = str(row['site_shown_uom']).strip().lower()

    # Special case: stück can appear as St in site_shown_uom
    if grammage_unit == 'stück':
        return 'stk' in site_shown_uom

    return grammage_unit in site_shown_uom

violations = loaded_data[~loaded_data.apply(is_valid, axis=1)]

print(
    violations[
        ['pdp_url', 'grammage_quantity', 'grammage_unit', 'site_shown_uom']
    ]
)

                                                                                           pdp_url  \
38528   https://www.coles.com.au/product/jameson-ultra-blend-dry-and-lime-can-250ml-pack-4-1687250   
38529   https://www.coles.com.au/product/jameson-ultra-blend-dry-and-lime-can-250ml-pack-4-1687250   
38530   https://www.coles.com.au/product/jameson-ultra-blend-dry-and-lime-can-250ml-pack-4-1687250   
38531   https://www.coles.com.au/product/jameson-ultra-blend-dry-and-lime-can-250ml-pack-4-1687250   
38532   https://www.coles.com.au/product/jameson-ultra-blend-dry-and-lime-can-250ml-pack-4-1687250   
...                                                                                            ...   
730950                https://www.coles.com.au/product/multix-aluminium-foil-wrap-30-meters-432978   
730951                https://www.coles.com.au/product/multix-aluminium-foil-wrap-30-meters-432978   
730952                https://www.coles.com.au/product/multix-aluminium-foil-wrap-

In [175]:
# Convert to string to avoid errors with nulls/numbers
loaded_data['Part number'] = loaded_data['Part number'].astype(str)
loaded_data['URL'] = loaded_data['URL'].astype(str)

# Find rows where part number is NOT present in URL
violations = loaded_data[
    ~loaded_data.apply(lambda row: row['Part number'] in row['URL'], axis=1)
]

print("URLs violating the requirement:")
for url in violations['URL']:
    print(url)

URLs violating the requirement:


In [248]:
# Validate grammage_quantity and grammage_unit
print("Grammage validation:\n")

valid_units = {
    'g', 'kg', 'mg', 'lb', 'oz', 'ml', 'l', 'ltr', 'litre', 'liter',
    'pcs', 'pc', 'pack', 'pkt', 'each', 'ea', 'count', 'unit', 'ct', 'kos', 'pz','cl','btl','stück','wg'
}

quantity_pattern = r'^\d+(\.\d+)?$'

if 'grammage_quantity' in loaded_data.columns:
    quantity = loaded_data['grammage_quantity'].astype(str).str.strip()
    invalid_quantity = loaded_data[~quantity.str.match(quantity_pattern, na=False)]
    invalid_comma = loaded_data[quantity.str.contains(',', na=False)]

    print(f"grammage_quantity rows with invalid format: {len(invalid_quantity)}")
    if len(invalid_quantity) > 0:
        print(invalid_quantity[['grammage_quantity','pdp_url']])

    print(f"grammage_quantity rows containing commas: {len(invalid_comma)}")
    if len(invalid_comma) > 0:
        print(invalid_comma[['grammage_quantity','pdp_url']])
    else:
        print("grammage_quantity column not found.")

    print()

if 'grammage_unit' in loaded_data.columns:
    unit = loaded_data['grammage_unit'].astype(str).str.strip().str.lower()
    invalid_unit = loaded_data[~unit.isin(valid_units) & ~loaded_data['grammage_unit'].isna()]

    print(f"grammage_unit rows with invalid units: {len(invalid_unit)}")
    if len(invalid_unit) > 0:
        print(invalid_unit[['grammage_unit','pdp_url']])
    else:
        print("grammage_unit values are valid.")
else:
    print("grammage_unit column not found.")

Grammage validation:

grammage_quantity rows with invalid format: 3984
        grammage_quantity  \
281                   NaN   
282                   NaN   
283                   NaN   
284                   NaN   
285                   NaN   
...                   ...   
761145                NaN   
761146                NaN   
761147                NaN   
761148                NaN   
761149                NaN   

                                                                                            pdp_url  
281                      https://www.coles.com.au/product/somma-somma-watermelon-and-lime-a-4309260  
282                      https://www.coles.com.au/product/somma-somma-watermelon-and-lime-a-4309260  
283                      https://www.coles.com.au/product/somma-somma-watermelon-and-lime-a-4309260  
284                      https://www.coles.com.au/product/somma-somma-watermelon-and-lime-a-4309260  
285                      https://www.coles.com.au/product/somma-somma-

In [309]:
loaded_data.duplicated().sum()

0

In [100]:
# Validate that site_shown_uom ends with grammage_unit; print unique_id for mismatches
print("Checking site_shown_uom endswith grammage_unit:\n")

unit_aliases = {'ltr': 'l', 'litre': 'l', 'liter': 'l'}
mismatched_ids = []

if 'site_shown_uom' not in loaded_data.columns:
    print("site_shown_uom column not found. Cannot perform check.")
elif 'grammage_unit' not in loaded_data.columns:
    print("grammage_unit column not found. Cannot perform check.")
else:
    for idx, row in loaded_data.iterrows():
        site_shown_uom = row.get('site_shown_uom')
        unit = row.get('grammage_unit')
        uid = row.get('unique_id', idx)

        if pd.isna(site_shown_uom) or pd.isna(unit):
            continue

        site_shown_uom_s = str(site_shown_uom).strip().lower()
        unit_s = str(unit).strip().lower()
        unit_norm = unit_aliases.get(unit_s, unit_s)

        # valid end forms: ' <unit>' or '<unit>' (no space)
        candidates = [f" {unit_s}", unit_s, f" {unit_norm}", unit_norm]

        if not any(site_shown_uom_s.endswith(c) for c in candidates):
            mismatched_ids.append(uid)

    print(f"Total rows checked: {len(loaded_data)}")
    print(f"Rows where site_shown_uom does NOT end with grammage_unit: {len(mismatched_ids)}")
    if mismatched_ids:
        print("Sample unique_ids with mismatches:", mismatched_ids[:50])
        # show example rows
        cols_show = [c for c in ['unique_id','site_shown_uom','grammage_quantity','grammage_unit'] if c in loaded_data.columns]
        print("\nExample mismatched rows:")
        print(loaded_data[loaded_data['unique_id'].isin(mismatched_ids)][cols_show].head(10).to_dict('records'))

Checking site_shown_uom endswith grammage_unit:

Total rows checked: 10291
Rows where site_shown_uom does NOT end with grammage_unit: 177
Sample unique_ids with mismatches: [4066447967920, 4066447369731, 4066447792911, 4066447792997, 9000101566697, 4009175968616, 4066447790382, 4066447877984, 4066447888393, 4066447369755, 4066447791204, 4066447791495, 4058172172199, 3830049641011, 4066447792935, 4058172622410, 4066447791211, 4101230033915, 4066447969870, 4066447791501, 4066447994759, 3830055190008, 3830055190022, 4070765003956, 4066447793222, 4066447345001, 4066447792188, 4066447792959, 4066447791235, 4066447484519, 4066447896886, 9000101582574, 9000101581478, 9000101590272, 9000101583328, 9000101581539, 9000101581874, 4067796212853, 4066447861297, 9000101582642, 9000101590128, 4058172308376, 4066447951325, 4067796070279, 4066447890006, 4067796174663, 4070765056297, 4066447345025, 4066447888317, 4066447989922]

Example mismatched rows:
[{'unique_id': 4066447967920, 'site_shown_uom': '4

In [257]:
# Validate product_name + site_shown_uom rule
print("Checking product_name/site_shown_uom -> grammage fields:\n")

required_cols = {'product_name', 'site_shown_uom', 'grammage_quantity', 'grammage_unit'}
missing_cols = sorted(required_cols - set(loaded_data.columns))

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    def has_uom_in_product(row):
        product_name = str(row['product_name']).strip().lower() if pd.notna(row['product_name']) else ''
        site_uom = str(row['site_shown_uom']).strip().lower() if pd.notna(row['site_shown_uom']) else ''
        return bool(site_uom) and site_uom in product_name

    violations = loaded_data[
        loaded_data.apply(
            lambda row: has_uom_in_product(row)
            and (
                str(row['grammage_quantity']).strip() == ''
                or str(row['grammage_unit']).strip() == ''
            ),
            axis=1
        )
    ]

    print(f"Rows violating the rule: {len(violations)}")

    if not violations.empty:
        print(violations[['unique_id', 'product_name', 'site_shown_uom', 'grammage_quantity', 'grammage_unit']].to_string(index=False))
    else:
        print("No violations found.")


Checking product_name/site_shown_uom -> grammage fields:

Rows violating the rule: 0
No violations found.


In [298]:
# Check rows where selling_price != regular_price
print("Price comparison: selling_price vs regular_price\n")

if 'selling_price' in loaded_data.columns and 'regular_price' in loaded_data.columns:
    mismatches = loaded_data[loaded_data['selling_price'] != loaded_data['regular_price']]
    
    print(f"Total rows: {len(loaded_data)}")
    print(f"Rows where selling_price ≠ regular_price: {len(mismatches)}")
    
    if len(mismatches) > 0:
        mismatched_ids = mismatches['unique_id'].unique() if 'unique_id' in loaded_data.columns else mismatches.index.tolist()
        print(f"\nUnique IDs with price mismatch ({len(mismatched_ids)} unique):")
        print(list(mismatched_ids[:100]))  # Show first 100
        
        if len(mismatched_ids) > 100:
            print(f"... and {len(mismatched_ids) - 100} more")
        
        # Show sample rows
        cols_show = [c for c in ['pdp_url', 'selling_price', 'regular_price','promotion_price','promotion_description'] if c in loaded_data.columns]
        print("\nSample mismatched rows:")
        print(mismatches[cols_show])
    else:
        print("✓ All rows have selling_price = regular_price")
else:
    if 'selling_price' not in loaded_data.columns:
        print("⚠ selling_price column not found.")
    if 'regular_price' not in loaded_data.columns:
        print("⚠ regular_price column not found.")

Price comparison: selling_price vs regular_price

Total rows: 761163
Rows where selling_price ≠ regular_price: 167583

Unique IDs with price mismatch (6454 unique):
[8127923, 4309260, 1783560, 8871359, 8440109, 8185668, 3723740, 1715424, 7391777, 4477581, 3734601, 8053838, 4476760, 1015423, 9102429, 9677715, 1501770, 1823289, 8334746, 7024030, 1753489, 3742326, 1603883, 8985846, 1591514, 6284589, 8643300, 5816092, 8949682, 3741980, 6538290, 2855600, 2296418, 4309838, 1756534, 6438172, 3797292, 4514859, 1141518, 3715833, 8551686, 1993595, 3320165, 7311916, 8058208, 3588838, 1700019, 5074060, 1141507, 5016482, 1840277, 2236370, 4505337, 1155578, 2390697, 9583087, 3615500, 1033093, 4223798, 4473161, 7019517, 8260333, 3706741, 6377602, 1141711, 8100155, 5300409, 5048219, 3660797, 1609879, 8392935, 1198548, 5430380, 3871941, 6823898, 1864990, 1976359, 354193, 1282826, 1855650, 1338567, 9066813, 8314716, 5255166, 7051112, 4805443, 8310249, 5856290, 6187141, 112365, 7645300, 8269087, 3270216,

In [297]:
# Check rows where promotion_price >= regular_price or selling_price > regular_price
# Convert price columns to numeric (if not already)
for col in ['regular_price', 'selling_price', 'promotion_price']:
    loaded_data[col] = pd.to_numeric(loaded_data[col], errors='coerce')

# Rows where promotion_price exists
promo_exists = loaded_data['promotion_price'].notna()

# Invalid rows
invalid_rows = loaded_data[
    (
        promo_exists &
        (loaded_data['promotion_price'] >= loaded_data['regular_price'])
    )
    |
    (
        loaded_data['selling_price'] > loaded_data['regular_price']
    )
]

print(f"Number of invalid rows: {len(invalid_rows)}")

print(
    invalid_rows[
        [
            'unique_id',
            'regular_price',
            'selling_price',
            'promotion_price',
            'pdp_url', 'promotion_description'
        ]
    ]
)

Number of invalid rows: 0
Empty DataFrame
Columns: [unique_id, regular_price, selling_price, promotion_price, pdp_url, promotion_description]
Index: []


In [110]:
# Check rows where promotion_description is empty but promotion_price has a value
print("Checking empty promotion description with promotion price:\n")

required_cols = {'promotion_description', 'promotion_price', 'unique_id', 'regular_price', 'selling_price', 'pdp_url'}
missing_cols = sorted(required_cols - set(loaded_data.columns))

if missing_cols:
    print(f"Missing columns: {missing_cols}")
else:
    violations = loaded_data[
        (loaded_data['promotion_description'].fillna('').astype(str).str.strip() == '')
        & (loaded_data['promotion_price'].fillna('').astype(str).str.strip() != '')
    ]

    print(f"Rows violating the rule: {len(violations)}")
    if not violations.empty:
        print("Violating rows")
        print(violations[['promotion_description', 'promotion_price', 'regular_price', 'selling_price', 'pdp_url']])
    else:
        print("No violations found.")


Checking empty promotion description with promotion price:

Rows violating the rule: 0
No violations found.


In [111]:
# Convert price columns to numeric
promotion_price = pd.to_numeric(
    loaded_data['promotion_price'],
    errors='coerce'
)

selling_price = pd.to_numeric(
    loaded_data['selling_price'],
    errors='coerce'
)

# promotion_price exists and is different from selling_price
invalid_rows = loaded_data[
    promotion_price.notna() &
    (promotion_price != selling_price)
]

print(f"Number of invalid rows: {len(invalid_rows)}")

print(
    invalid_rows[
        [
            'unique_id',
            'promotion_price',
            'selling_price',
            'regular_price',
            'promotion_description',
            'pdp_url'
        ]
    ]
)

Number of invalid rows: 0
Empty DataFrame
Columns: [unique_id, promotion_price, selling_price, regular_price, promotion_description, pdp_url]
Index: []


In [341]:
# Records violating the rule:
# If promotion_price is not null, both promotion_valid_from
# and promotion_valid_upto must be non-empty

violations = loaded_data[
    loaded_data['promotion_price'].notna() &
    (
        loaded_data['promotion_valid_from'].isna() |
        loaded_data['promotion_valid_upto'].isna() |
        (loaded_data['promotion_valid_from'].astype(str).str.strip() == '') |
        (loaded_data['promotion_valid_upto'].astype(str).str.strip() == '')
    )
]

print(
    violations[
        [
            'unique_id',
            'promotion_price',
            'promotion_valid_from',
            'promotion_valid_upto'
        ]
    ]
)

           unique_id  promotion_price  promotion_valid_from  \
89           5733460             1.99                   NaN   
166          4069836             7.99                   NaN   
641          5733439             1.99                   NaN   
780    2020002925284             2.19                   NaN   
783    2020004262615             0.49                   NaN   
...              ...              ...                   ...   
41052  2020004808608             1.49                   NaN   
41058         391009             3.79                   NaN   
41115  2020004762849             1.49                   NaN   
41127        8197467             1.99                   NaN   
41135        6441531             0.67                   NaN   

       promotion_valid_upto  
89                      NaN  
166                     NaN  
641                     NaN  
780                     NaN  
783                     NaN  
...                     ...  
41052                   NaN  
410

In [559]:
# MERCATOR requirement: Expected product_unique_key
expected_key = loaded_data['unique_id'].astype(str) + 'P'

# Check mismatches
invalid_rows = loaded_data[
    loaded_data['product_unique_key'].astype(str) != expected_key
]

print(f"Number of mismatched rows: {len(invalid_rows)}")

# Print the mismatched rows
print(invalid_rows[['unique_id', 'product_unique_key']])

Number of mismatched rows: 0
Empty DataFrame
Columns: [unique_id, product_unique_key]
Index: []


In [39]:
# MERCATOR requirement: Identify rows where "Trajno znižano" is present anywhere
mask = loaded_data['promotion_description'].str.contains(
    'Trajno znižano',
    # 'Znižano',
    case=False,
    na=False
)

# Invalid rows
invalid_rows = loaded_data[
    mask &
    (
        # regular_price and selling_price are not equal
        (loaded_data['regular_price'] != loaded_data['selling_price'])
        |
        # price_was is not empty
        (
            loaded_data['price_was'].notna() &
            (loaded_data['price_was'].astype(str).str.strip() != "")
        )
        |
        # promotion_price is not empty
        (
            loaded_data['promotion_price'].notna() &
            (loaded_data['promotion_price'].astype(str).str.strip() != "")
        )
    )
]

print(f"Number of invalid rows: {len(invalid_rows)}")

# Print the entire invalid rows
print(invalid_rows)

Number of invalid rows: 0
Empty DataFrame
Columns: [unique_id, competitor_name, store_name, store_addressline1, store_addressline2, store_suburb, store_state, store_postcode, store_addressid, extraction_date, product_name, brand, brand_type, grammage_quantity, grammage_unit, drained_weight, producthierarchy_level1, producthierarchy_level2, producthierarchy_level3, producthierarchy_level4, producthierarchy_level5, producthierarchy_level6, producthierarchy_level7, regular_price, selling_price, price_was, promotion_price, promotion_valid_from, promotion_valid_upto, promotion_type, percentage_discount, promotion_description, package_sizeof_sellingprice, per_unit_sizedescription, price_valid_from, price_per_unit, multi_buy_item_count, multi_buy_items_price_total, currency, breadcrumb, pdp_url, variants, product_description, instructions, storage_instructions, preparationinstructions, instructionforuse, country_of_origin, allergens, age_of_the_product, age_recommendations, flavour, nutrition

In [37]:
# MERCATOR requirement: Find rows where "Pika" appears anywhere in promotion_description
mask = loaded_data['promotion_description'].str.contains(
    'Pika',
    case=False,
    na=False
)

# Find violations
invalid_rows = loaded_data[
    mask &
    (
        # promotion_price is not empty
        (
            loaded_data['promotion_price'].notna() &
            (loaded_data['promotion_price'].astype(str).str.strip() != "")
        )
        |
        # regular_price and selling_price are not equal
        (
            loaded_data['regular_price'] != loaded_data['selling_price']
        )
    )
]

print(f"Number of invalid rows: {len(invalid_rows)}")

# Print entire invalid rows
print(invalid_rows)

Number of invalid rows: 0
Empty DataFrame
Columns: [unique_id, competitor_name, store_name, store_addressline1, store_addressline2, store_suburb, store_state, store_postcode, store_addressid, extraction_date, product_name, brand, brand_type, grammage_quantity, grammage_unit, drained_weight, producthierarchy_level1, producthierarchy_level2, producthierarchy_level3, producthierarchy_level4, producthierarchy_level5, producthierarchy_level6, producthierarchy_level7, regular_price, selling_price, price_was, promotion_price, promotion_valid_from, promotion_valid_upto, promotion_type, percentage_discount, promotion_description, package_sizeof_sellingprice, per_unit_sizedescription, price_valid_from, price_per_unit, multi_buy_item_count, multi_buy_items_price_total, currency, breadcrumb, pdp_url, variants, product_description, instructions, storage_instructions, preparationinstructions, instructionforuse, country_of_origin, allergens, age_of_the_product, age_recommendations, flavour, nutrition

In [234]:
# Check product_name and site_shown_uom relationship: product_name should end with site_shown_uom
# Find records where product_name does NOT end with site_shown_uom
# violations = loaded_data[
#     loaded_data.apply(
#         lambda row: (
#             pd.notna(row['site_shown_uom']) and
#             str(row['site_shown_uom']).strip() != '' and
#             not str(row['product_name']).strip().endswith(
#                 str(row['site_shown_uom']).strip()
#             )
#         ),
#         axis=1
#     )
# ]

# # Print required columns
# print(
#     violations[
#         ['unique_id', 'product_name', 'site_shown_uom']
#     ]
# )


# Find records where site_shown_uom is not contained in product_name
violations = loaded_data[
    loaded_data.apply(
        lambda row: (
            pd.notna(row['site_shown_uom']) and
            str(row['site_shown_uom']).strip() != '' and
            str(row['site_shown_uom']).strip().lower()
            not in str(row['product_name']).strip().lower()
        ),
        axis=1
    )
]

# Print required columns
print(
    violations[
        ['unique_id', 'product_name', 'site_shown_uom']
    ]
)

       unique_id                                      product_name  \
0         483846  BIO ČEBULA RUMENA, SPAR NATUR*PUR, PAKIRANO, 1KG   
1         482756             BIO RDEČA ČEBULA SPAR NATUR*PUR, 500G   
2         656842                   ČESEN, S-BUDGET, 400G, PAKIRANO   
3         652735               ČEBULA RDEČA V TUBI, PAKIRANO, 500G   
4         160593                              BELA ČEBULA, TEHTANO   
...          ...                                               ...   
16633     607257                    VLOŽEK OL-9, PREMIUM, 9-DNEVNI   
16634     638399                                            NAPAKA   
16635     340755   NAGROBNA SVEČA SPIRALA VELIKA RDEČA, ILKOS, 1/1   
16636     280116           STENJ ZA VEČNO LUČ, VEČERNICO, PAX, 5/1   
16637     649699  RIŽEV KIS ZA SUSHI BREZ GLUTENA, KIKKOMAN, 125ML   

      site_shown_uom  
0               1 KG  
1             0.5 KG  
2             0.4 KG  
3             0.5 KG  
4               1 KG  
...              ... 

In [278]:
# If site_shown_uom contains:Teebeutel Packung, Teebeutel, Beutel, Btl, Teebeutel Karton, Teebeutel Paket, Packung Beutel
# Then: grammage_unit must be "btl"
# stück

# Values that indicate the product is sold as a bag/pack
uom_values = [
    # "Teebeutel Packung",
    # "Teebeutel",
    # "Beutel",
    # "Btl",
    # "Teebeutel Karton",
    # "Teebeutel Paket",
    # "Packung Beutel"
    # "Portion Packung", "ANW"
    "wg", "Waschgänge" , "Waschgang"
]

# Check whether site_shown_uom contains any of the specified values
uom_condition = loaded_data['site_shown_uom'].fillna('').astype(str).str.lower().apply(
    lambda x: any(value.lower() in x for value in uom_values)
)

# Violations: matching UOM description but grammage_unit is not 'btl'
violations = loaded_data[
    uom_condition &
    (
        loaded_data['grammage_unit'].isna() |
        (loaded_data['grammage_unit'].astype(str).str.strip().str.lower() != 'wg')
    )
]

# Print relevant columns
print(
    violations[
        ['unique_id', 'site_shown_uom', 'grammage_unit', 'pdp_url']
    ]
)
print(violations['grammage_unit'].unique())

          unique_id site_shown_uom grammage_unit  \
2668  4015200038162          60 wg            ml   

                                                              pdp_url  
2668  https://www.dm.at/p/d/3130473/somat-geschirrreiniger-gel-5-in-1  
['ml']


In [42]:
# Filter records where producthierarchy_level3 contains 'Fleisch & Fisch'
# Select rows where producthierarchy_level3 contains 'Fleisch & Fisch'
meat_fish = loaded_data[
    (loaded_data['producthierarchy_level3']
    .fillna('')
    .str.contains('Fleisch & Fisch', case=False, na=False)) 
    &
    ~(loaded_data['producthierarchy_level4']
    .fillna('')
    .str.contains('Charcuterie & Wurstwaren', case=False, na=False))
]

# Check whether site_shown_uom is contained in price_per_unit
violations = meat_fish[
    ~meat_fish.apply(
        lambda row: (
            pd.notna(row['site_shown_uom']) and
            pd.notna(row['price_per_unit']) and
            str(row['site_shown_uom']).strip().lower()
            in str(row['price_per_unit']).strip().lower()
        ),
        axis=1
    )
]

print(
    violations[
        [
            'unique_id',
            'producthierarchy_level3',
            'producthierarchy_level4',
            'site_shown_uom',
            'grammage_quantity',
            'grammage_unit',
            'regular_price',
            'selling_price',
            'price_per_unit'
        ]
    ]
)

KeyError: "None of [Index(['unique_id', 'producthierarchy_level3', 'producthierarchy_level4',\n       'site_shown_uom', 'grammage_quantity', 'grammage_unit', 'regular_price',\n       'selling_price', 'price_per_unit'],\n      dtype='object')] are in the [columns]"

In [211]:
# if regular price in data - promotion price is selling price 

violations = loaded_data[
    loaded_data['regular_price'].notna() &
    (
        loaded_data['promotion_price'].isna() |
        (loaded_data['promotion_price'] != loaded_data['selling_price'])
    )
]

print(
    violations[
        [
            'unique_id',
            'regular_price',
            'promotion_price',
            'selling_price', 'pdp_url'
        ]
    ]
)

      unique_id  regular_price  promotion_price  selling_price  \
0         80220           2.79              NaN           2.79   
1         82345           2.79              NaN           2.79   
2         82761           1.95              NaN           1.95   
3         80505           1.75              NaN           1.75   
4         28673           2.99              NaN           2.99   
...         ...            ...              ...            ...   
3615     156012           1.29              NaN           1.29   
3616     153772           2.99              NaN           2.99   
3617      67886           1.59              NaN           1.59   
3618    5111668           3.49              NaN           3.49   
3619    1016070           1.29              NaN           1.29   

                                                                                                         pdp_url  
0         https://sortiment.lidl.ch/de/catalog/product/view/id/13737/s/schweizer-aepfel-rot-

In [228]:


violations = loaded_data[
    loaded_data['promotion_description'].notna() &
    (
        loaded_data['promotion_valid_from'].isna() |
        loaded_data['promotion_valid_upto'].isna()
    )
]

print(
    violations[
        [
            'unique_id',
            'promotion_description',
            'promotion_valid_from',
            'promotion_valid_upto'
        ]
    ]
)

      unique_id promotion_description promotion_valid_from  \
5        151217                  -20%            13.8.2026   
18       149516                  -20%            13.8.2026   
30      5107879                  -20%            13.8.2026   
34      5114255                Aktion                  NaN   
35      1801686                Aktion                  NaN   
...         ...                   ...                  ...   
3323   10057218        +1 Dose gratis            13.8.2026   
3324   10057215                   XXL            13.8.2026   
3326   10057217                   XXL            13.8.2026   
3335   10057240                Aktion            13.8.2026   
3340   10057202                   XXL            13.8.2026   

     promotion_valid_upto  
5                     NaN  
18                    NaN  
30                    NaN  
34                    NaN  
35                    NaN  
...                   ...  
3323                  NaN  
3324                  NaN  
332

In [230]:
# Find records where variants contains an unescaped |

violations = loaded_data[
    loaded_data['product_name']
    .fillna('')
    .astype(str)
    .str.contains(r'(?<!\\)\|', regex=True, na=False)
]

print(violations[['pdp_url']])

Empty DataFrame
Columns: [pdp_url]
Index: []


In [238]:
# Find PDP URLs linked to multiple product names
url_product_counts = (
    loaded_data.groupby('pdp_url')['product_name']
    .nunique()
    .reset_index(name='product_name_count')
)

# Keep only PDP URLs with more than one product name
inconsistent_urls = url_product_counts[
    url_product_counts['product_name_count'] > 1
]

# Display all corresponding records
result = loaded_data[
    loaded_data['pdp_url'].isin(inconsistent_urls['pdp_url'])
][['pdp_url', 'product_name']].drop_duplicates()

print(result.to_string(index=False))

Empty DataFrame
Columns: [pdp_url, product_name]
Index: []


In [236]:
# Find '|' that is NOT preceded by '\'
invalid_mask = loaded_data.astype(str).apply(
    lambda col: col.str.contains(r'(?<!\\)\|', regex=True, na=False)
)

# Rows containing at least one invalid '|'
invalid_rows_mask = invalid_mask.any(axis=1)

# Get the invalid rows
invalid_rows = loaded_data.loc[invalid_rows_mask]

# Columns containing invalid '|'
invalid_columns = invalid_mask.columns[invalid_mask.any()].tolist()

# Print only unique_id, pdp_url, and the invalid columns
columns_to_print = ['unique_id', 'pdp_url'] + invalid_columns

print("Number of invalid rows:", len(invalid_rows))

print("\nRows violating the rule:")
print(loaded_data.loc[invalid_rows_mask, columns_to_print])

Number of invalid rows: 0

Rows violating the rule:
Empty DataFrame
Columns: [unique_id, pdp_url]
Index: []


In [232]:
# Dates validation YYYY.MM.DD format and future dates

from datetime import datetime

date_columns = ['promotion_valid_from', 'promotion_valid_upto']

today = pd.Timestamp.today().normalize()

# Store invalid/future date masks for each column
invalid_date_mask = pd.DataFrame(False, index=loaded_data.index, columns=date_columns)
future_date_mask = pd.DataFrame(False, index=loaded_data.index, columns=date_columns)

for col in date_columns:

    # Treat NaN and empty strings as valid/ignored
    non_empty = (
        loaded_data[col].notna() &
        loaded_data[col].astype(str).str.strip().ne('')
    )

    # Convert dates using the required format YYYY.MM.DD
    parsed_dates = pd.to_datetime(
        loaded_data[col].where(non_empty),
        format='%d.%m.%Y',
        errors='coerce'
    )

    # Invalid date:
    # Non-empty value that could not be parsed
    invalid_date_mask[col] = non_empty & parsed_dates.isna()

    # Future date
    future_date_mask[col] = non_empty & parsed_dates.gt(today)


# Rows containing either invalid or future dates
violating_rows_mask = (
    invalid_date_mask.any(axis=1) |
    future_date_mask.any(axis=1)
)

# Print only unique_id and pdp_url
violating_rows = loaded_data.loc[
    violating_rows_mask,
    ['unique_id', 'pdp_url','promotion_valid_from', 'promotion_valid_upto']
]

print("Number of rows violating the date rule:", len(violating_rows))
print(violating_rows)

Number of rows violating the date rule: 254
      unique_id  \
3071   10056916   
3076   10057062   
3078   10056917   
3079   10057058   
3080   10056752   
...         ...   
3483   10057352   
3484   10057361   
3485   10057302   
3486   10057355   
3487   10057354   

                                                                                  pdp_url  \
3071                             https://www.lidl.ch/p/de-CH/italiamo-croissant/p10056916   
3076  https://www.lidl.ch/p/de-CH/qualite-suisse-terra-natura-baguette-rustique/p10057062   
3078                              https://www.lidl.ch/p/de-CH/italiamo-cornetti/p10056917   
3079                                   https://www.lidl.ch/p/de-CH/deluxe-coppa/p10057058   
3080         https://www.lidl.ch/p/de-CH/livarno-premium-vorhangschal-set-2-tlg/p10056752   
...                                                                                   ...   
3483                                https://www.lidl.ch/p/de-CH/zenker-backf

In [129]:
# Check starting_from column: if not empty, must start with 'AED'

invalid_rows = loaded_data[
    loaded_data['starting_from'].notna() &
    (loaded_data['starting_from'].astype(str).str.strip() != '') &
    ~loaded_data['starting_from'].astype(str).str.match(r'^AED\b')
]

print(f"Number of invalid rows: {len(invalid_rows)}")

print(invalid_rows[['url', 'starting_from']])

Number of invalid rows: 0
Empty DataFrame
Columns: [url, starting_from]
Index: []


In [128]:
# Check starting_from column: if not empty, must end with 'M'

invalid_rows = loaded_data[
    loaded_data['starting_from'].notna() &
    (loaded_data['starting_from'].astype(str).str.strip() != '') &
    ~loaded_data['starting_from'].astype(str).str.strip().str.endswith('M') &
    ~loaded_data['starting_from'].astype(str).str.strip().str.endswith('K')
]

print(f"Number of invalid rows: {len(invalid_rows)}")

print(invalid_rows[['url', 'starting_from']])

Number of invalid rows: 0
Empty DataFrame
Columns: [url, starting_from]
Index: []


In [293]:
# checking whether details end with 'sqft'
column = 'breadcrumb'
violations = loaded_data[
    loaded_data[column].astype(str).str.strip().str.lower().str.contains('> >', na=False) 
]

print(violations[[column, 'pdp_url']])

Empty DataFrame
Columns: [breadcrumb, pdp_url]
Index: []
